## Dataset_description : Predictive Maintenance Dataset

This dataset contains time-series operational data collected from industrial machines,
designed to support predictive maintenance and failure prediction tasks.

The dataset includes 24,042 records and 15 features, representing machine sensor readings,
operational conditions, and maintenance-related information.

Each row corresponds to a machine observation recorded at a specific timestamp.
The data reflects real-world conditions, including missing values in several sensor columns.

The dataset supports multiple analytical use cases such as:
- Failure prediction (classification)
- Remaining useful life estimation (regression)
- Anomaly detection
- Maintenance optimization

It includes both input features (sensor and operational data) and target variables
related to machine failure and maintenance outcomes.

## Data Dictionary

| Column Name              | Description                                                                 |
|-------------------------|-----------------------------------------------------------------------------|
| timestamp               | Date and time of the machine observation                                   |
| machine_id              | Unique identifier for each machine                                         |
| machine_type            | Type or category of the machine                                            |
| vibration_rms           | Root Mean Square (RMS) of vibration indicating mechanical condition        |
| temperature_motor       | Motor temperature measured in degrees Celsius                              |
| current_phase_avg       | Average electrical current across phases (machine load indicator)          |
| pressure_level          | Pressure measurement within the system                                     |
| rpm                     | Rotational speed in revolutions per minute                                 |
| operating_mode          | Current operating state of the machine (e.g., idle, active)                |
| hours_since_maintenance | Number of hours since last maintenance activity                            |
| ambient_temp            | Ambient environmental temperature                                          |
| rul_hours               | Remaining Useful Life (RUL) in hours before expected failure               |
| failure_within_24h      | Indicates whether a failure occurs within 24 hours (1 = yes, 0 = no)       |
| failure_type            | Type or category of machine failure                                        |
| estimated_repair_cost   | Estimated cost required to repair the machine after failure                |

## Show some data info

In [2]:
# from google.colab import files
# uploaded = files.upload()

In [3]:
import pandas as pd
df = pd.read_csv("predictive_maintenance_v3.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'predictive_maintenance_v3.csv'

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

## Fixing Data

In [ ]:
"""
We have a timestamp column that contains both date and time.
We need to split it into two separate columns: one for the date and one for the time,
ensuring that each column has the appropriate and correct data type for further analysis.
"""

df['timestamp'] = pd.to_datetime(df['timestamp'])

df['date'] = df['timestamp'].dt.date
df['time'] = df['timestamp'].dt.time

In [ ]:
df['machine_id'] = df['machine_id'].astype('object')

### Check Nulls

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
# Filling missing values in numerical columns with 0 to handle nulls and ensure consistency for analysis
df.fillna(0, inplace=True)

In [ ]:
# Check duplicates
df.duplicated().sum()


In [ ]:
df['failure'] = df['failure_within_24h'].astype(int)
display(df[['failure_within_24h', 'failure']].head())

## Check Outliers

In [ ]:
def detect_outliers_iqr(df, cols):
    df = df.copy()
    outlier_mask = pd.DataFrame(False, index=df.index, columns=cols)

    for col in cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        outlier_mask[col] = (
            (df[col] < Q1 - 1.5 * IQR) |
            (df[col] > Q3 + 1.5 * IQR)
        )

    return outlier_mask

In [ ]:
num_cols = ['vibration_rms', 'pressure_level', 'temperature_motor', 'current_phase_avg', 'rpm']

outliers = detect_outliers_iqr(df, num_cols)

In [ ]:
outliers.sum()

In [ ]:
df[outliers.any(axis=1)]

# Outliers detection using the IQR method across numerical columns.
# The summary shows the number of detected outliers per feature,
# and the filtered dataset highlights rows where at least one feature
# lies outside the acceptable range (1.5 * IQR rule).
# A total of 1787 rows were flagged as containing outliers for further analysis.

In [ ]:
# Check if there any wrong values
df[df['rpm'] < 0]
df[df['temperature_motor'] < 0]

In [ ]:
# Standardize Categorical Values
df['operating_mode'] = df['operating_mode'].str.lower().str.strip()
df['machine_type'] = df['machine_type'].str.upper().str.strip()

# **Data Analysis **

In [ ]:
## Failure Rate
failure_rate = df['failure_within_24h'].mean()
print("Failure Rate:", failure_rate)

In [ ]:
## Count Failed VS not Failed
df['failure_within_24h'].value_counts()

In [ ]:
## Average conditions when failure happens
df.groupby('failure_within_24h')[[
    'vibration_rms',
    'temperature_motor',
    'rpm',
    'hours_since_maintenance'
]].mean()

In [ ]:
##highest failure Machine
df[df['failure_within_24h'] == 1]['machine_type'].value_counts()

In [ ]:
## Relationship between failure and temprature
df[['temperature_motor', 'failure_within_24h']].corr()

In [ ]:
## Relationship between failure and vibration
df[['vibration_rms', 'failure_within_24h']].corr()

In [ ]:
## mentainance delays and failure
df.groupby('failure_within_24h')['hours_since_maintenance'].mean()

In [ ]:
## Average remaining useful life
df[df['failure_within_24h'] == 1]['rul_hours'].mean()

In [ ]:
## Average repair cost
df[df['failure_within_24h'] == 1]['estimated_repair_cost'].mean()

In [ ]:
## Failure Type
df['failure_type'].value_counts()

In [ ]:
## correlation between all variable
df.corr(numeric_only=True)

In [ ]:
## Relationship between operating mode and failure
df.groupby('operating_mode')['failure_within_24h'].mean()

In [ ]:
## sum Machine type with highest repair cost
df.groupby('machine_type')['estimated_repair_cost'].sum().sort_values(ascending=False).head()

In [ ]:
## Machine Id with highest repair cost
df.groupby('machine_id')['estimated_repair_cost'].sum().sort_values(ascending=False).head()

In [ ]:
## when failure happened
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.set_index('timestamp')['failure_within_24h'].resample('D').sum()

In [ ]:
#Number of machine with high risk
high_risk = df[
    (df['temperature_motor'] > 70) &
    (df['vibration_rms'] > 3) &
    (df['hours_since_maintenance'] > 200)
]

len(high_risk)

## Machine Learning for Failure Prediction

This section introduces a basic Machine Learning model to predict machine failure within 24 hours. The goal is to demonstrate how predictive analytics can support maintenance strategies, focusing on a straightforward implementation suitable for data analysis projects. We will use a Random Forest Classifier due to its balance of performance and interpretability.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Define target column
target_column = 'failure_within_24h'
y = df[target_column]

# Define features by dropping target, identifiers, and already processed/future-related columns
feature_columns = df.drop(
    columns=[
        target_column,
        'timestamp',
        'date',
        'time',
        'machine_id',
        'rul_hours',
        'failure_type',
        'estimated_repair_cost'
    ]
).columns
X = df[feature_columns]

# Identify numerical and categorical features
numerical_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include='object').columns.tolist()

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Create a column transformer for preprocessing numerical and categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)

# Apply preprocessing to training and testing data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)


In [ ]:
# Initialize and train the Random Forest Classifier with default parameters
model = RandomForestClassifier(random_state=42)
model.fit(X_train_processed, y_train)

print("Random Forest Classifier trained successfully.")

In [ ]:
# Make predictions on the preprocessed test set
y_pred = model.predict(X_test_processed)

print("Predictions made on the test set.")

In [ ]:
# Calculate Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# Confusion Matrix Plot
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted No Failure', 'Predicted Failure'],
            yticklabels=['Actual No Failure', 'Actual Failure'])
plt.title('Confusion Matrix for Random Forest Classifier')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
# Get feature names after one-hot encoding
encoded_feature_names = preprocessor.get_feature_names_out()

# Create a DataFrame for feature importances
feature_importances = pd.DataFrame({'Feature': encoded_feature_names, 'Importance': model.feature_importances_})
feature_importances = feature_importances.sort_values(by='Importance', ascending=False)

# Plot top 10 feature importances
plt.figure(figsize=(10, 7))
sns.barplot(x='Importance', y='Feature', data=feature_importances.head(10), palette='viridis', hue='Feature', legend=False)
plt.title('Top 10 Feature Importances from Random Forest')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## Business Interpretation & Conclusion

**Business Interpretation:** The Random Forest model demonstrates a good capability to predict machine failures within 24 hours, with an accuracy of approximately 96.5% and a solid F1-score for the failure class. This suggests it can be a valuable tool for proactive maintenance. The confusion matrix indicates that the model is reasonably effective at identifying actual failures while keeping false alarms manageable. Key features such as `temperature_motor`, `rpm`, and `vibration_rms` are identified as most important, reinforcing their critical role in machine health monitoring. Implementing this model can help maintenance teams prioritize interventions, reduce unplanned downtime, and optimize resource allocation.

**Conclusion:** This analysis highlights the potential of using machine learning to enhance predictive maintenance strategies. The Random Forest model provides a reliable method for early detection of potential failures, allowing for a shift from reactive to proactive maintenance. Future work could explore optimizing monitoring thresholds, integrating real-time data streams, and exploring the impact of more diverse failure data.

In [ ]:
df.to_csv('processed_predictive_maintenance_v3.csv', index=False)